In [0]:
from pyspark.sql import SparkSession, functions as F
import builtins, sys

# Ensure spark is accessible by module-level code in edtech_gold_utils
builtins.spark = SparkSession.builder.getOrCreate()
sys.modules.pop("edtech_gold_utils", None)

from edtech_gold_utils import (
    TABLES,
    require_table,
    has_col,
    col_or_null,
    pct_norm,
    overwrite_delta_table,
)

In [0]:
# ── Read all source tables ──────────────────────────────────────────
eng  = spark.read.table("edtech_project.edtech_silver.student_course_engagement")
enr  = spark.read.table("edtech_project.edtech_silver.enrollment_risk_profile")
disc = spark.read.table("edtech_project.edtech_silver.discussion_posts_parsed")
enroll = spark.read.table("edtech_project.edtech_bronze.enrollments_bronze")

spark.sql("CREATE SCHEMA IF NOT EXISTS edtech_project.edtech_gold")
print("Source tables loaded ✓")

In [0]:
# ── Engagement metrics per course ───────────────────────────────────
# active_students, avg_quiz_score, avg_video_completion_pct, avg_engagement_score

eng_base = (
    eng
    .withColumn("quiz_pct", F.col("quiz_pass_rate") / 100.0)
    .withColumn("video_pct", F.col("video_completion_rate") / 100.0)
    .withColumn(
        "engagement_score",
        0.6 * F.coalesce(F.col("quiz_pct"), F.lit(0.0))
        + 0.4 * F.coalesce(F.col("video_pct"), F.lit(0.0)),
    )
)

eng_metrics = (
    eng_base
    .groupBy("course_id")
    .agg(
        F.countDistinct(
            F.when(F.col("last_active_date") == F.current_date(), F.col("student_id"))
        ).alias("active_students"),
        F.round(F.avg("quiz_pass_rate"), 2).alias("avg_quiz_score"),
        F.round(F.avg("video_completion_rate"), 2).alias("avg_video_completion_pct"),
        F.round(F.avg("engagement_score"), 4).alias("avg_engagement_score"),
    )
)

print(f"eng_metrics: {eng_metrics.count()} courses")

In [0]:
# ── Enrollment metrics per course ───────────────────────────────────
# completion_rate, new_enrollments, dropout_count

enroll_metrics = (
    enroll
    .groupBy("course_id")
    .agg(
        # completion_rate = completed / total enrollments
        F.round(
            F.sum(F.when(F.col("status") == "COMPLETED", 1).otherwise(0))
            / F.count("*"),
            4,
        ).alias("completion_rate"),
        # new_enrollments: students enrolled today
        F.sum(
            F.when(F.to_date(F.col("enrolled_date")) == F.current_date(), 1).otherwise(0)
        ).cast("long").alias("new_enrollments"),
        # dropout_count: students currently in DROPPED status
        # Note: no status_change_date column available — this is a snapshot count,
        # not strictly "moved to DROPPED today". Consider adding CDC for true daily deltas.
        F.sum(
            F.when(F.col("status") == "DROPPED", 1).otherwise(0)
        ).cast("long").alias("dropout_count"),
    )
)

print(f"enroll_metrics: {enroll_metrics.count()} courses")

In [0]:
# ── Discussion metrics per course ───────────────────────────────────
# instructor_response_rate = instructor replies / total student posts

disc_metrics = (
    disc
    .groupBy("course_id")
    .agg(
        F.sum(
            F.when(
                (F.col("is_instructor_post") == True)
                & F.col("parent_post_id").isNotNull(),
                1,
            ).otherwise(0)
        ).alias("_instr_replies"),
        F.sum("is_student_post").alias("_student_posts"),
    )
    .withColumn(
        "instructor_response_rate",
        F.round(
            F.when(
                F.col("_student_posts") > 0,
                F.col("_instr_replies") / F.col("_student_posts"),
            ).otherwise(0.0),
            4,
        ),
    )
    .select("course_id", "instructor_response_rate")
)

print(f"disc_metrics: {disc_metrics.count()} courses")

In [0]:
# ── Assemble final gold table and write to Delta ────────────────────
GOLD_TABLE = "edtech_project.edtech_gold.course_performance_daily"

gold = (
    eng_metrics
    .join(enroll_metrics, "course_id", "full_outer")
    .join(disc_metrics, "course_id", "full_outer")
    .withColumn("report_date", F.current_date())
    .select(
        "course_id",
        "report_date",
        F.coalesce(F.col("active_students"), F.lit(0)).alias("active_students"),
        F.coalesce(F.col("completion_rate"), F.lit(0.0)).alias("completion_rate"),
        F.coalesce(F.col("avg_quiz_score"), F.lit(0.0)).alias("avg_quiz_score"),
        F.coalesce(F.col("avg_video_completion_pct"), F.lit(0.0)).alias("avg_video_completion_pct"),
        F.coalesce(F.col("new_enrollments"), F.lit(0)).cast("long").alias("new_enrollments"),
        F.coalesce(F.col("dropout_count"), F.lit(0)).cast("long").alias("dropout_count"),
        F.coalesce(F.col("avg_engagement_score"), F.lit(0.0)).alias("avg_engagement_score"),
        F.coalesce(F.col("instructor_response_rate"), F.lit(0.0)).alias("instructor_response_rate"),
    )
)

# Write partitioned by report_date
(
    gold.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("report_date")
    .saveAsTable(GOLD_TABLE)
)

print(f"✓ Wrote {gold.count()} rows to {GOLD_TABLE}")

In [0]:
# ── Z-ORDER by course_id for fast per-course queries ────────────────
spark.sql(f"OPTIMIZE {GOLD_TABLE} ZORDER BY (course_id)")
print(f"✓ OPTIMIZE ZORDER complete on {GOLD_TABLE}")

In [0]:
# ── Verify gold table ───────────────────────────────────────────────
result = spark.read.table(GOLD_TABLE)
print(f"Rows: {result.count()}, Partitions: report_date")
result.printSchema()
display(result.orderBy("course_id"))

In [0]:
enr.printSchema()

In [0]:
disc.printSchema()